Q1. Embedding a query

In [3]:
from embedder import Embedder

2026-06-25 10:04:40.647378765 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [4]:
model = Embedder()

v = model.encode( "How does approximate nearest neighbor search work?")

print(v[0])

-0.02058203437252893


Loading the data


In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Q2. Cosine similarity

In [6]:
import numpy as np

model = Embedder()

query = "How does approximate nearest neighbor search work?"
q = model.encode(query)

doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

v = model.encode(doc['content'])

# similarity = query_vector @ page_vector

similarity = np.dot(q , v)
print(similarity)



0.36107027225589694


Q3. Chunking and search by hand - A chunk is a smaller piece of a document.


In [7]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
# extract text
texts = [c['content'] for c in chunks]

#embed
vector = model.encode_batch(texts)

len(vector)


295

In [9]:
vector.shape


(295, 384)

create matrix

In [10]:
X = np.array(vector)

scores = X.dot(q)

The largest score is the most relevant chunk.

In [11]:
best_idx = scores.argmax()
best_idx

np.int64(94)

In [12]:
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

Q4. Vector search with minsearch

In [13]:
from minsearch import VectorSearch


for chunk, vectors in zip(chunks, vector):
    chunk['vectors'] = vectors


index = VectorSearch(
    keyword_fields=["vectors"]
)

index.fit(X, chunks)

In [14]:
query = "What metric do we use to evaluate a search engine?"

q = model.encode(query)

results = index.search(query_vector=q , num_results=5)

results



[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [15]:
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


Q5. Text search vs vector search

In [16]:
from minsearch import Index

kindex = Index(
    text_fields=["content"]
)

kindex.fit(chunks)



query = "How do I store vectors in PostgreSQL?"

q = model.encode(query)



vector search

In [17]:
vector_results = index.search(query_vector=q, num_results=5)

for r in vector_results:
    print(r['filename'])


02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


text search

In [18]:
keyword_results = kindex.search(
    query=query,
    num_results=5
)


for f in keyword_results:
    print(f['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


Q6. Hybrid search

In [19]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [21]:
query = "How do I give the model access to tools?"

q = model.encode(query)

vector_results = index.search(query_vector=q, num_results=5)

text_results = kindex.search(query=query, num_results=5)

In [24]:
hybrid_results = rrf([vector_results, text_results])
hybrid_results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'